In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, chisquare

np.random.seed(42)
base_temp = 11.5   # середньорічна температура варіанта 3 (Одеса)
amplitude = 13     # сезонна амплітуда варіанта 3
city = "Одеса"


def season(month):
    if month in (12, 1, 2):
        return "зима"
    if month in (3, 4, 5):
        return "весна"
    if month in (6, 7, 8):
        return "літо"
    return "осінь"


rows = []
for year in [2021, 2022, 2023, 2024]:
    for month in range(1, 13):
        seasonal = amplitude * np.cos((month - 7) / 12 * 2 * np.pi)
        noise = np.random.normal(0, 1.0)
        temp = round(base_temp + seasonal + noise, 1)
        diff = temp - base_temp
        if diff < -3:
            norm_cat = "холодніше"
        elif diff > 3:
            norm_cat = "тепліше"
        else:
            norm_cat = "звичайно"
        rows.append({
            "місто": city, "рік": year, "місяць": month,
            "температура": temp, "сезон": season(month),
            "відхилення_від_норми": norm_cat,
        })

climate = pd.DataFrame(rows)   # 48 рядків, 4 роки x 12 місяців
climate.head(12)
print(climate.shape)
climate.groupby('сезон')['температура'].agg(['mean', 'min', 'max'])

In [ ]:
seasons_order = ["зима", "весна", "літо", "осінь"]
norm_order = ["холодніше", "звичайно", "тепліше"]

table = pd.crosstab(climate["сезон"], climate["відхилення_від_норми"]).reindex(
    index=seasons_order, columns=norm_order
)
print("Фактичні частоти:")
print(table)
print("\nСумарно по рядках:", table.sum(axis=1).to_dict())
print("Сумарно по стовпцях:", table.sum(axis=0).to_dict())
print("Загальний N =", table.values.sum())

In [ ]:
table_norm_index = pd.crosstab(
    climate["сезон"], climate["відхилення_від_норми"], normalize="index"
).reindex(index=seasons_order, columns=norm_order)
print("Частки усередині сезону (normalize='index'):")
print(table_norm_index.round(3))
print("\nКонтроль суми по рядках:", table_norm_index.sum(axis=1).round(6).to_dict())

In [ ]:
chi2, pvalue, dof, expected = chi2_contingency(table)
print(f"chi2 = {chi2:.4f}")
print(f"p-value = {pvalue:.3e}")
print(f"dof = {dof}")

expected_df = pd.DataFrame(expected, index=table.index, columns=table.columns)
print("\nОчікувані частоти:")
print(expected_df.round(2))

print("\nФактичні (спостережені):")
print(table)

diff = table - expected_df
print("\nРозбіжність observed - expected:")
print(diff.round(2))

print("\nВнесок кожної клітинки в chi2 (O-E)^2/E:")
print((diff ** 2 / expected_df).round(2))

In [ ]:
print("Матриця expected повторно:\n", expected_df.round(2), "\n")
low_cells = [(idx, col, round(expected_df.loc[idx, col], 2))
               for idx in expected_df.index for col in expected_df.columns
               if expected_df.loc[idx, col] < 5]
print("Клітинки з expected < 5:", len(low_cells))
for cell in low_cells:
    print("   ", cell)
print("\nМінімальна очікувана частота:", round(expected_df.values.min(), 2))
print("Умова застосовності виконана?", bool(expected_df.values.min() >= 5))

In [ ]:
# Варіант A: «не тепліше» = холодніше + звичайно
climate_a = climate.copy()
climate_a["категорія"] = np.where(
    climate_a["відхилення_від_норми"] == "холодніше", "не тепліше", "тепліше"
)
table_a = pd.crosstab(climate_a["сезон"], climate_a["категорія"]).reindex(
    index=seasons_order, columns=["не тепліше", "тепліше"]
)
print("Об'єднана таблиця A:\n", table_a)
chi2_a, p_a, dof_a, exp_a = chi2_contingency(table_a)
print(f"chi2 = {chi2_a:.4f}, p-value = {p_a:.3e}, dof = {dof_a}")
print("expected A:\n", pd.DataFrame(exp_a, index=table_a.index, columns=table_a.columns).round(2))

In [ ]:
# Варіант B: «не холодніше» = звичайно + тепліше
climate_b = climate.copy()
climate_b["категорія"] = np.where(
    climate_b["відхилення_від_норми"] == "холодніше", "холодніше", "не холодніше"
)
table_b = pd.crosstab(climate_b["сезон"], climate_b["категорія"]).reindex(
    index=seasons_order, columns=["холодніше", "не холодніше"]
)
print("Об'єднана таблиця B:\n", table_b)
chi2_b, p_b, dof_b, exp_b = chi2_contingency(table_b)
print(f"chi2 = {chi2_b:.4f}, p-value = {p_b:.3e}, dof = {dof_b}")
print("expected B:\n", pd.DataFrame(exp_b, index=table_b.index, columns=table_b.columns).round(2))

In [ ]:
season_counts = climate["сезон"].value_counts().reindex(seasons_order)
print("Кількість спостережень у сезонах:\n", season_counts)

n_total = len(climate)
expected_seasons = [n_total * 0.25] * 4
print("Очікувані частоти при рівномірній чверті:", expected_seasons)

res_season = chisquare(f_obs=season_counts.values, f_exp=expected_seasons)
print(f"chi2 = {res_season.statistic:.4f}, p-value = {res_season.pvalue:.4f}")

In [ ]:
observed_counts = climate["відхилення_від_норми"].value_counts().reindex(norm_order)
print("Фактичні частоти:\n", observed_counts)

# --- H0 (власне припущення): ---
# «звичайно» — 50 %, «холодніше» — 25 %, «тепліше» — 25 %.
# Обґрунтування нижче: поріг |відхилення| > 3 °C перевищує рівно близько половини
# місяців, бо сезонний доданок ±13*cos(...) залишає |відхилення| у смузі ±3 °C
# приблизно для 6 із 12 місяців (середня амплітуда близько половини 13 °C).
my_prop = {"холодніше": 0.25, "звичайно": 0.50, "тепліше": 0.25}
expected_my = [n_total * my_prop[k] for k in norm_order]
print("Заявлена пропорція H0:", my_prop)
print("Очікувані частоти:", expected_my)

res_prop = chisquare(f_obs=observed_counts.values, f_exp=expected_my)
print(f"chi2 = {res_prop.statistic:.4f}, p-value = {res_prop.pvalue:.3e}")

In [ ]:
# Для порівняння: ще кілька правдоподібних гіпотез
for name, prop in [
    ("25/50/25", [0.25, 0.50, 0.25]),
    ("20/60/20", [0.20, 0.60, 0.20]),
    ("рівні третини", [1/3, 1/3, 1/3]),
    ("0.33/0.34/0.33", [0.33, 0.34, 0.33]),
]:
    r = chisquare(observed_counts.values, [n_total * p for p in prop])


## Контрольні питання

### 1. Чому в χ² саме квадрат відхилення і саме ділення на очікувану часту?
$\chi^2=\sum\frac{(O-E)^2}{E}$. Квадрат потрібен, бо (а) знак відхичення не має значення — перевищення й недобір однаково свідчать про зв'язок, (б) квадрат знімає лінійність і перетворює розподіл сум незалежних нормальних величин на χ² (теорема Вільямса). Ділення на E потрібне для **нормалізації варіації**: абсолютне відхилення у клітинці з очікуваною частотою 20 у 2 рази інформативніше за таке саме відхилення в клітинці з очікуваною частотою 2, бо абсолютна дисперія-count пропорційна E (для біноміальних даних Var = E). Без ділення на E клітинки з малими E «здували» б статистику. Результат — кожна клітинка дає безрозмірний внесок, який можна скласти.

### 2. Як рахується очікувана частота і чому ця формула коректна?
$E_{ij}=\frac{R_i\cdot C_j}{N}$. При H0 (незалежності) ймовірність потрапити в рядок $i$ дорівнює $R_i/N$ (маргінальна), у стовпець $j$ — $C_j/N$, а за незалежності добуток дає $P_{ij}=(R_i/N)(C_j/N)$. Добуток $N$ спостережень із цією ймовірністю дає $E_{ij}=(R_i/N)(C_j/N)\cdot N$. Отже формула — це прямий наслідок припущення незалежності, і χ² вимірює саме відхилення факту від цього очікування. У моєму наборі $E=12\cdot20/48=5$ і $12\cdot8/48=2$ — бачимо, чому частина клітинок мала очікувані частоти.

### 3. «p-value велике» ≠ «доведено незалежність»
p-value велике означає лише, що **немає підстав відхилити H₀**: ймовірність отримати такі або навіть більш «нетипові» дані за H₀ достатньо велика. Це не довід про відсутність зв'язку. Потужність критерію залежить від N і розміру ефекту: за N = 48 навіть помірний зв'язок можна не виявити. Тому формулювання коректне: «не знайдено статистично значущого зв'язку на рівні α = 0.05», а не «змінні незалежні».

### 4. Що робити, якщо очікувана частота < 5 і чому це проблема?
Це проблема, бо апроксимація χ²-розподілу для сум квадратів нормальних величин вимагає достатньої кількості очікуваних випадків у клітинці. Для E < 5 (а при 2×2 — E < 10) розподіл статистики сильно асиметричний і має «важчий» правий хвіст, тому реальне p-value систематично завищується (критерій дає занадто завищений p — ризик помилки I типу зростає, тобто можна помилково не знайти зв'язок). Рішення: (1) об'єднати категорії в змістові групи (тут — «не тепліше»), (2) застосувати точний тест Фішера, (3) збільшити вибірку. У моєму випадку після об'єднання всі E ≥ 5 і p-value лишився мізерним, тож висновок стійкий.

### Підсумок
- Побудовано `pd.crosstab` для `сезон` × `відхилення_від_норми` та його нормалізовані версії.
- Критерій незалежності: χ² = 38.40, p ≈ 9.38·10⁻⁷, df = 6 → зв'язок **підтверджено** (очікувано, бо змінні походять зі спільної формули).
- Умова застосовності порушена (4 клітинки з E = 2.0); після об'єднання «холодніше» + «звичайно» → «не тепліше»: χ² = 26.06, p ≈ 9.28·10⁻⁶, усі E ≥ 5, висновок не змінився.
- Критерій узгодженості для сезонів: χ² = 0.0, p = 1.0 → розподіл точно рівномірний за конструкцією.
- Власна пропорція 25/50/25 для відхилень: χ² = 21.33, p ≈ 2.33·10⁻⁵ → H₀ **відхиляємо**; фактично «звичайно» лише 8 випадків (17 %).

**Варіант 3 — Одеса.**